# E-Commerce Analytics: Anàlisi de Cohorts i Modelatge de CLV (Customer Lifetime Value)
**Autor:** Oriol Anguera Milà

**Eines:** Python (Pandas, Numpy), SQL, DuckDB, Power BI.

**Objectiu del Projecte:** Demostrar l'aplicació pràctica de l'estadística i l'enginyeria de dades per resoldre problemes reals de negoci. Aquest repositori integra la simulació de comportament de consumidors mitjançant distribucions matemàtiques (Log-normal i Exponencial) amb el modelatge de dades en SQL.


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. Generació de Dades (Python)

Construirem el nostre propi entorn de dades. Simularem el comportament d'un e-commerce real entre 2024 i 2025 aplicant distribucions probabilístiques:

* **Fidelitat i Recurrència (Distribució Exponencial):** Modelitzem el concepte de *churn* i la Llei de Pareto. La majoria de clients fan 1 o 2 compres, mentre que una minoria fa compres recurrents.
* **Import de la Cistella (Distribució Log-normal):** Aquesta distribució genera de forma realista moltes compres de valor mitjà-baix i unes poques compres d'alt valor.

Això ens garanteix un dataset net, escalable i matemàticament robust per posar a prova l'anàlisi de cohorts.

In [ ]:
np.random.seed(42)
n_customers = 3000
start_date = pd.to_datetime('2024-01-01')
end_date = pd.to_datetime('2025-12-31')

# 1. GENERAR CLIENTS
customer_ids = np.arange(1, n_customers + 1)
random_days = np.random.randint(0, (end_date - start_date).days, n_customers)
join_dates = start_date + pd.to_timedelta(random_days, unit='d') # Data d'alta aleatòria entre 2024 i finals de 2025
channels = np.random.choice(['Organic', 'Google Ads', 'Meta', 'Referral'], n_customers, p=[0.4, 0.3, 0.2, 0.1]) # Canal de contacte

customers_df = pd.DataFrame({'customer_id': customer_ids, 'join_date': join_dates, 'channel': channels})

# 2. GENERAR COMPRES
orders = []
order_id_counter = 1

for _, client in customers_df.iterrows():
    n_orders = int(np.random.exponential(scale=1.5)) + 1 # Nombre de compres que farà aquest client (distribució exponencial)

    current_date = client['join_date']

    for _ in range(n_orders):
        amount = round(np.random.lognormal(mean=3.5, sigma=0.8), 2) # Preu de la compra (distribució lognormal)

        if current_date <= end_date:
            orders.append({'order_id': order_id_counter, 'customer_id': client['customer_id'], 'order_date': current_date, 'amount': amount})
            order_id_counter += 1

        days_to_next_order = int(np.random.exponential(scale=45)) # Mitjana de 45 dies entre compres, en cas de fer-ne més d'una
        current_date += timedelta(days=days_to_next_order)

orders_df = pd.DataFrame(orders)

# 3. GUARDAR ELS FITXERS
customers_df.to_csv('customers.csv', index=False)
orders_df.to_csv('orders.csv', index=False)

✅ S'han generat 3000 clients i 5865 transaccions.
Els fitxers 'customers.csv' i 'orders.csv' estan a punt.


In [ ]:
customers_df.head()

,customer_id,join_date,channel
0,1,2024-04-12,Referral
1,2,2025-03-11,Organic
2,3,2024-09-27,Meta
3,4,2024-04-16,Referral
4,5,2024-03-12,Referral


In [ ]:
orders_df.head()

,order_id,customer_id,order_date,amount
0,1,1,2024-04-12,22.88
1,2,2,2025-03-11,46.02
2,3,3,2024-09-27,66.73
3,4,4,2024-04-16,22.80
4,5,4,2024-05-09,68.57


# 2. Transformació i Modelatge de Dades (SQL)

Preparem les dades per construir una Matriu de Retenció. Mitjançant DuckDB, apliquem consultes SQL per calcular:

1. **Cohort del Client:** El mes de la seva primera interacció.
2. **Mes de la Transacció:** El mes en què ocorre cada comanda.
3. **Cohort Index:** La diferència en mesos entre la transacció i la creació de la cohort, necessari per veure l'evolució de la retenció al llarg del temps.

In [ ]:
!pip install duckdb

import duckdb

In [ ]:
# Trobem la data de la primera compra de cada client (la seva Cohort), creuem les transaccions amb la cohort del client, arrodonim la data al mes i calculem quants mesos han passat des de la primera compra (Cohort Index)
query_cohorts = """
WITH cohort_setup AS (
    SELECT
        customer_id,
        DATE_TRUNC('month', MIN(order_date)) AS cohort_month
    FROM orders_df
    GROUP BY customer_id
),

order_details AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.amount,
        DATE_TRUNC('month', o.order_date) AS order_month,
        c.cohort_month
    FROM orders_df o
    JOIN cohort_setup c ON o.customer_id = c.customer_id
)

SELECT
    *,
    DATE_DIFF('month', cohort_month, order_month) AS cohort_index
FROM order_details
ORDER BY customer_id, order_month;
"""

cohorts_base_df = duckdb.query(query_cohorts).to_df()

display(cohorts_base_df.head(10))

,order_id,customer_id,amount,order_month,cohort_month,cohort_index
0,1,1,22.88,2024-04-01,2024-04-01,0
1,2,2,46.02,2025-03-01,2025-03-01,0
2,3,3,66.73,2024-09-01,2024-09-01,0
3,4,4,22.80,2024-04-01,2024-04-01,0
4,5,4,68.57,2024-05-01,2024-04-01,1
5,6,5,10.26,2024-03-01,2024-03-01,0
6,7,6,9.05,2025-12-01,2025-12-01,0
7,8,6,25.52,2025-12-01,2025-12-01,0
8,9,7,29.68,2024-01-01,2024-01-01,0
9,10,8,45.29,2025-09-01,2025-09-01,0


Ara que cada comanda té el seu `cohort_index`, agruparem les dades per calcular el volum de clients. Per a cada combinació de **Mes de Cohort**, **Índex de Cohort** i **Canal d'adquisició**, calcularem:
1. El total de clients únics que van comprar en aquell període.
2. La suma total de la facturació, que ens servirà més endavant per al modelatge del CLV.

Aquesta taula resultant serà la font de dades per fer la Matriu de Retenció a Power BI.

In [ ]:
query_matriu_agregada = """
SELECT
    c.cohort_month,
    c.cohort_index,
    cust.channel,
    COUNT(DISTINCT c.customer_id) AS active_customers,
    SUM(c.amount) AS total_revenue
FROM cohorts_base_df c
JOIN customers_df cust ON c.customer_id = cust.customer_id
GROUP BY c.cohort_month, c.cohort_index, cust.channel
ORDER BY c.cohort_month, c.cohort_index, cust.channel;
"""

matriu_agregada_df = duckdb.query(query_matriu_agregada).to_df()

display(matriu_agregada_df.head(15))

,cohort_month,cohort_index,channel,active_customers,total_revenue
0,2024-01-01,0,Google Ads,37,2070.66
1,2024-01-01,0,Meta,28,1697.39
2,2024-01-01,0,Organic,61,3172.87
3,2024-01-01,0,Referral,16,912.63
4,2024-01-01,1,Google Ads,12,913.19
5,2024-01-01,1,Meta,3,181.72
6,2024-01-01,1,Organic,12,579.09
7,2024-01-01,1,Referral,7,475.86
8,2024-01-01,2,Google Ads,8,532.91
9,2024-01-01,2,Meta,5,335.94


# 3. Exportació per a Visualització

La taula "matriu_agregada_df" té tota la informació necessària per crear la Matriu de Retenció amb Power BI. L'exportem com a fitxer csv:

In [ ]:
matriu_agregada_df.to_csv('cohort_matrix_data.csv', index=False)